In [49]:
language = 'pt'

# 1. Gravação de Áudio Com Python (e Uma Pitada de JavaScript) 🎤

In [50]:
from IPython.display import Javascript, display
from google.colab import output
from base64 import b64decode

def gravar_audio():
    display(Javascript("""
    async function recordAudio() {
      const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
      const recorder = new MediaRecorder(stream);
      let chunks = [];

      recorder.ondataavailable = e => chunks.push(e.data);

      recorder.start();

      await new Promise(resolve => setTimeout(resolve, 5000)); // 5 segundos

      recorder.stop();

      await new Promise(resolve => recorder.onstop = resolve);

      const blob = new Blob(chunks);
      const arrayBuffer = await blob.arrayBuffer();
      const base64Audio = btoa(String.fromCharCode(...new Uint8Array(arrayBuffer)));

      return base64Audio;
    }

    recordAudio().then(audio => {
      google.colab.kernel.invokeFunction('notebook.salvar_audio', [audio], {});
    });
    """))

    def salvar_audio(base64_audio):
        audio_bytes = b64decode(base64_audio)
        with open("gravacao.wav", "wb") as f:
            f.write(audio_bytes)
        print("✅ Áudio gravado com sucesso!")

    output.register_callback("notebook.salvar_audio", salvar_audio)

print("👉 Execute esta célula para gravar o áudio!")

👉 Execute esta célula para gravar o áudio!


In [51]:
gravar_audio()

<IPython.core.display.Javascript object>

✅ Áudio gravado com sucesso!


# 2. Reconhecimento de Fala com Whisper (OpenAI) 🧠

In [52]:
!pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [53]:
import os
os.environ["OPENAI_API_KEY"] = "SUA_API_KEY_AQUI"

In [54]:
from google.colab import userdata
from openai import OpenAI
import os

# Pega a chave de forma segura
api_key = userdata.get("OPENAI_API_KEY")

if not api_key:
    raise Exception("❌ API KEY não encontrada nos Secrets do Colab!")

client = OpenAI(api_key=api_key)

# Verifica se o arquivo existe
if not os.path.exists("gravacao.wav"):
    raise Exception("❌ Arquivo de áudio não encontrado! Execute a etapa 1 primeiro.")

# Abre o áudio
with open("gravacao.wav", "rb") as audio_file:
    transcricao = client.audio.transcriptions.create(
        model="whisper-1",
        file=audio_file
    )

texto_usuario = transcricao.text

print("🗣️ Você disse:")
print(texto_usuario)

🗣️ Você disse:
Crie uma assistente virtual para auxiliar no suporte ao cliente.


In [55]:
!apt-get install ffmpeg -y

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [56]:
!ffmpeg -i gravacao.webm gravacao.wav

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

# 3. Integração com a API do ChatGPT 💬

In [57]:
!pip install openai

In [58]:
if not texto_usuario:
    raise Exception("❌ Nenhum texto foi transcrito.")

resposta = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": texto_usuario}
    ]
)

texto_resposta = resposta.choices[0].message.content

print("\n🤖 Resposta da IA:")
print(texto_resposta)


🤖 Resposta da IA:
Claro! Abaixo estão as etapas e exemplos de como você pode estruturar uma assistente virtual para suporte ao cliente. Vou incluir sugestões de funcionalidades, diálogos de exemplo e tecnologias que você pode usar.

### Estrutura da Assistente Virtual

#### 1. **Definição de Funções**
   - **Atendimento ao Cliente:** Responder perguntas frequentes (FAQ).
   - **Resolução de Problemas:** Auxiliar na solução de problemas técnicos.
   - **Informação sobre Produtos/Serviços:** Fornecer detalhes sobre produtos ou serviços.
   - **Acompanhamento de Pedidos:** Permitir que os usuários verifiquem o status de seus pedidos.
   - **Coleta de Feedback:** Solicitar feedback sobre o atendimento ou produtos.

#### 2. **Plataformas e Tecnologias**
   - **Chatbots:** Use ferramentas como ChatGPT, Dialogflow, ou Microsoft Bot Framework.
   - **Integração com CRM:** Conectar a assistente a sistemas de CRM como Salesforce ou HubSpot.
   - **Aplicações de Mensagem:** Implementar em plataf

# 4. Sintetizando a Resposta do ChatGPT Como Voz (gTTS) 🔊

In [59]:
!pip install gTTS

In [60]:
from gtts import gTTS
from IPython.display import Audio

if not texto_resposta:
    raise Exception("❌ Nenhuma resposta gerada.")

tts = gTTS(texto_resposta, lang='pt-br')
tts.save("resposta.mp3")

print("\n🔊 Tocando resposta em áudio:")
Audio("resposta.mp3")


🔊 Tocando resposta em áudio:
